# Dataframe Consolidation: Three-Table Architecture

**Purpose:** Combine four source dataframes into three specialized tables:
1. `dim_districts` - Geographic dimension (one row per district)
2. `fact_monthly_climate` - Monthly weather + vegetation
3. `fact_seasonal_crop` - Seasonal crop yields + aggregated weather features

**Key Challenge:** Handle temporal granularity mismatch and year-wrapping seasons (Rabi)

**Reference:** See plan at `.claude/plans/given-the-four-dataframe-replicated-puppy.md`

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Display options
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

print("✓ Imports complete")

In [ ]:
# Paths
current = Path.cwd()
project_root = current if (current / "CLAUDE.md").exists() else current.parent.parent

pickle_folder_path = project_root / "data" / "processed"

print(f"Project root: {project_root}")
print(f"Data path: {pickle_folder_path}")

## Load Source Data

In [ ]:
print("Loading cleaned source dataframes (from 03_data_quality.ipynb)...\n")

# Load consolidated, cleaned files
print("1. Loading crop_yield...")
crop_yield_df = pd.read_pickle(pickle_folder_path / "crop_yield.pkl")
print(f"   Loaded: {len(crop_yield_df):,} rows")

print("2. Loading geography...")
geography_df = pd.read_pickle(pickle_folder_path / "geography.pkl")
print(f"   Loaded: {len(geography_df):,} rows")

print("3. Loading precipitation...")
precipitation_df = pd.read_pickle(pickle_folder_path / "precipitation.pkl")
print(f"   Loaded: {len(precipitation_df):,} rows")

print("4. Loading vegetation...")
vegetation_df = pd.read_pickle(pickle_folder_path / "vegetation.pkl")
print(f"   Loaded: {len(vegetation_df):,} rows")

print("\n✓ All source data loaded (already cleaned and consolidated)")

In [ ]:
# Prepare data for consolidation
print("Preparing data for consolidation...\n")

# Extract integer year from datetime (for joining)
crop_yield_df["crop_year"] = crop_yield_df["year"].dt.year

print("✓ Data preparation complete")

In [ ]:
# Inspect source data structure
print("Source Data Structure:")
print("=" * 80)

print("\nCrop Yield:")
print(f"  Shape: {crop_yield_df.shape}")
print(
    f"  States: {crop_yield_df['state'].nunique()}, Districts: {crop_yield_df['district'].nunique()}"
)
print(
    f"  Years: {crop_yield_df['crop_year'].min()} - {crop_yield_df['crop_year'].max()}"
)
print(f"  Columns: {list(crop_yield_df.columns)}")

print("\nPrecipitation:")
print(f"  Shape: {precipitation_df.shape}")
print(
    f"  States: {precipitation_df['state'].nunique()}, Districts: {precipitation_df['district'].nunique()}"
)
print(f"  Years: {precipitation_df['year'].min()} - {precipitation_df['year'].max()}")
print(f"  Columns: {list(precipitation_df.columns)}")

print("\nVegetation:")
print(f"  Shape: {vegetation_df.shape}")
print(
    f"  States: {vegetation_df['state'].nunique()}, Districts: {vegetation_df['district'].nunique()}"
)
print(f"  Years: {vegetation_df['year'].min()} - {vegetation_df['year'].max()}")
print(f"  Columns: {list(vegetation_df.columns)}")

print("\nGeography:")
print(f"  Shape: {geography_df.shape}")
print(
    f"  States: {geography_df['state'].nunique()}, Districts: {geography_df['district'].nunique()}"
)
print(f"  Note: Multiple rows per district (polygon boundary points)")

## Table 1: Create District Dimension

Aggregate geography to one row per district with computed centroids.

In [9]:
print("Creating dim_districts...\n")

# Geography is already aggregated at district level (from 03_data_quality.ipynb)
# Just add district_id and reorder columns
dim_districts = geography_df.copy()

# Create district_id (simple concatenation for readability)
dim_districts["district_id"] = (
    dim_districts["state"].astype(str) + "_" + dim_districts["district"].astype(str)
)

# Reorder columns
dim_districts = dim_districts[
    ["district_id", "state", "district", "centroid_lat", "centroid_lon"]
]

print(f"✓ dim_districts created: {len(dim_districts):,} rows")
print(f"\nSample:")
print(dim_districts.head())

Creating dim_districts...

✓ dim_districts created: 515 rows

Sample:
                    district_id           state       district  centroid_lat  \
0       Andhra Pradesh_Adilabad  Andhra Pradesh       Adilabad     19.284514   
1      Andhra Pradesh_Anantapur  Andhra Pradesh      Anantapur     14.312066   
2       Andhra Pradesh_Chittoor  Andhra Pradesh       Chittoor     13.331093   
3       Andhra Pradesh_Cuddapah  Andhra Pradesh       Cuddapah     14.490289   
4  Andhra Pradesh_East Godavari  Andhra Pradesh  East Godavari     16.782718   

   centroid_lon  
0     78.813212  
1     77.460158  
2     78.927639  
3     78.696139  
4     82.243207  


## Table 2: Create Monthly Climate Table

Outer join precipitation and vegetation to capture all months from both sources.

In [12]:
precipitation_df["year"].drop_duplicates()

0      2016
7      2017
19     2018
31     2019
43     2020
55     2021
67     2022
79     2023
91     2024
103    2025
115    2026
Name: year, dtype: int32

In [13]:
print("Creating fact_monthly_climate...\n")

# Outer join to capture all district-months from both sources
fact_monthly_climate = precipitation_df.merge(
    vegetation_df,
    on=["state", "district", "year", "month"],
    how="inner",
    indicator="data_source",
)

# Add date column for better time handling
fact_monthly_climate["date"] = pd.to_datetime(
    fact_monthly_climate[["year", "month"]].assign(day=1)
)

# Reorder columns
fact_monthly_climate = fact_monthly_climate[
    [
        "state",
        "district",
        "year",
        "month",
        "date",
        "precipitation_mm",
        "raw_ndvi_mean",
        "crop_area_km2",
        "valid_crop_area_km2",
        "valid_crop_fraction",
        "quality_status",
        "data_source",
    ]
]

# Sort by location and time
fact_monthly_climate = fact_monthly_climate.sort_values(
    ["state", "district", "year", "month"]
).reset_index(drop=True)

print(f"✓ fact_monthly_climate created: {len(fact_monthly_climate):,} rows")

# Check data source breakdown
print(f"\nData source breakdown:")
print(fact_monthly_climate["data_source"].value_counts())
print(f"\nSample:")
print(fact_monthly_climate.head(10))

Creating fact_monthly_climate...

✓ fact_monthly_climate created: 55,587 rows

Data source breakdown:
data_source
both          55587
left_only         0
right_only        0
Name: count, dtype: int64

Sample:
            state   district  year  month       date  precipitation_mm  \
0  Andhra Pradesh  Anantapur  2016      6 2016-06-01             57.98   
1  Andhra Pradesh  Anantapur  2016      7 2016-07-01             37.28   
2  Andhra Pradesh  Anantapur  2016      8 2016-08-01             40.20   
3  Andhra Pradesh  Anantapur  2016      9 2016-09-01             71.59   
4  Andhra Pradesh  Anantapur  2016     10 2016-10-01             36.11   
5  Andhra Pradesh  Anantapur  2016     11 2016-11-01              8.86   
6  Andhra Pradesh  Anantapur  2016     12 2016-12-01              8.09   
7  Andhra Pradesh  Anantapur  2017      1 2017-01-01              0.30   
8  Andhra Pradesh  Anantapur  2017      2 2017-02-01              0.05   
9  Andhra Pradesh  Anantapur  2017      3 2017-03-0

## Table 3: Create Seasonal Crop Table (Complex)

Aggregate monthly weather/vegetation to seasonal level and handle year-wrapping seasons.

**Key Challenge:** Rabi season spans Nov Year N → Mar Year N+1

In [14]:
# Define aggregation function
def aggregate_season_weather(row, precipitation_df, vegetation_df):
    """
    Aggregate weather/vegetation for months in a crop season.
    Uses season_months list (already computed in 03_data_quality.ipynb).
    Handles year-wrapping seasons like Rabi (Nov-Mar).
    """
    state = row["state"]
    district = row["district"]
    crop_year = row["crop_year"]
    season_months = row[
        "season_months"
    ]  # Already contains correct month list from 03_data_quality

    # Handle year wrapping: months >= 6 belong to crop_year, months <= 5 belong to crop_year+1
    # This aligns with agricultural calendar (Jun-Dec in year N, Jan-May in year N+1)
    early_months = [m for m in season_months if m >= 6]  # Jun-Dec of crop_year
    late_months = [m for m in season_months if m <= 5]  # Jan-May of crop_year+1

    # Filter precipitation for this season (handling year-wrapping)
    precip_season = precipitation_df[
        (precipitation_df["state"] == state)
        & (precipitation_df["district"] == district)
        & (
            (
                (precipitation_df["year"] == crop_year)
                & (precipitation_df["month"].isin(early_months))
            )
            | (
                (precipitation_df["year"] == crop_year + 1)
                & (precipitation_df["month"].isin(late_months))
            )
        )
    ]

    # Aggregate precipitation features
    seasonal_precip_total = (
        precip_season["precipitation_mm"].sum() if len(precip_season) > 0 else np.nan
    )
    seasonal_precip_avg = (
        precip_season["precipitation_mm"].mean() if len(precip_season) > 0 else np.nan
    )
    seasonal_precip_max = (
        precip_season["precipitation_mm"].max() if len(precip_season) > 0 else np.nan
    )
    rainy_days = (
        (precip_season["precipitation_mm"] > 2.5).sum()
        if len(precip_season) > 0
        else np.nan
    )

    # Filter vegetation for this season (handling year-wrapping)
    veg_season = vegetation_df[
        (vegetation_df["state"] == state)
        & (vegetation_df["district"] == district)
        & (
            (
                (vegetation_df["year"] == crop_year)
                & (vegetation_df["month"].isin(early_months))
            )
            | (
                (vegetation_df["year"] == crop_year + 1)
                & (vegetation_df["month"].isin(late_months))
            )
        )
    ]

    # Aggregate vegetation features
    seasonal_ndvi_mean = (
        veg_season["raw_ndvi_mean"].mean() if len(veg_season) > 0 else np.nan
    )
    seasonal_ndvi_min = (
        veg_season["raw_ndvi_mean"].min() if len(veg_season) > 0 else np.nan
    )
    seasonal_ndvi_max = (
        veg_season["raw_ndvi_mean"].max() if len(veg_season) > 0 else np.nan
    )

    # Phase-specific NDVI (early, mid, late thirds of season)
    n_months = len(season_months)
    if n_months >= 3 and len(veg_season) > 0:
        # Split season into thirds based on original season_months order
        early_months_subset = season_months[: n_months // 3]
        mid_months_subset = season_months[n_months // 3 : 2 * n_months // 3]
        late_months_subset = season_months[2 * n_months // 3 :]

        early_ndvi = veg_season[veg_season["month"].isin(early_months_subset)][
            "raw_ndvi_mean"
        ].mean()
        mid_ndvi = veg_season[veg_season["month"].isin(mid_months_subset)][
            "raw_ndvi_mean"
        ].mean()
        late_ndvi = veg_season[veg_season["month"].isin(late_months_subset)][
            "raw_ndvi_mean"
        ].mean()
    else:
        early_ndvi = mid_ndvi = late_ndvi = seasonal_ndvi_mean

    return pd.Series(
        {
            "seasonal_precip_total": seasonal_precip_total,
            "seasonal_precip_avg": seasonal_precip_avg,
            "seasonal_precip_max": seasonal_precip_max,
            "rainy_days": rainy_days,
            "seasonal_ndvi_mean": seasonal_ndvi_mean,
            "seasonal_ndvi_min": seasonal_ndvi_min,
            "seasonal_ndvi_max": seasonal_ndvi_max,
            "early_season_ndvi": early_ndvi,
            "mid_season_ndvi": mid_ndvi,
            "late_season_ndvi": late_ndvi,
        }
    )


print("✓ Aggregation function defined (uses season_months from 03_data_quality)")

✓ Aggregation function defined (uses season_months from 03_data_quality)


In [15]:
print("Creating fact_seasonal_crop...")
print("This will take several minutes due to row-by-row aggregation...\n")

# Apply aggregation to each crop row
import time

start_time = time.time()

weather_features = crop_yield_df.apply(
    lambda row: aggregate_season_weather(row, precipitation_df, vegetation_df), axis=1
)

elapsed = time.time() - start_time
print(f"✓ Aggregation complete in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")

Creating fact_seasonal_crop...
This will take several minutes due to row-by-row aggregation...

✓ Aggregation complete in 735.3 seconds (12.3 minutes)


In [16]:
# Combine with crop data
fact_seasonal_crop = pd.concat([crop_yield_df, weather_features], axis=1)

# Add geographic features from dim_districts
fact_seasonal_crop = fact_seasonal_crop.merge(
    dim_districts[["state", "district", "centroid_lat", "centroid_lon"]],
    on=["state", "district"],
    how="left",
)

# Select and order final columns
fact_seasonal_crop = fact_seasonal_crop[
    [
        "state",
        "district",
        "crop_year",
        "crop_name",
        "crop_type",
        "season",
        "area",
        "production",
        "yield",
        "season_start_month",
        "season_end_month",
        "seasonal_precip_total",
        "seasonal_precip_avg",
        "seasonal_precip_max",
        "rainy_days",
        "seasonal_ndvi_mean",
        "seasonal_ndvi_min",
        "seasonal_ndvi_max",
        "early_season_ndvi",
        "mid_season_ndvi",
        "late_season_ndvi",
        "centroid_lat",
        "centroid_lon",
    ]
]

print(f"\n✓ fact_seasonal_crop created: {len(fact_seasonal_crop):,} rows")
print(f"\nSample:")
print(fact_seasonal_crop.head())


✓ fact_seasonal_crop created: 361,385 rows

Sample:
            state   district  crop_year     crop_name    crop_type  season  \
0  Andhra Pradesh  Anantapur       1997     Arhar/Tur       Pulses  Kharif   
1  Andhra Pradesh  Anantapur       1997         Bajra      Cereals  Kharif   
2  Andhra Pradesh  Anantapur       1997   Castor Seed     Oilseeds  Kharif   
3  Andhra Pradesh  Anantapur       1997  Cotton(Lint)  Fiber Crops  Kharif   
4  Andhra Pradesh  Anantapur       1997  Dry Chillies       Spices  Kharif   

      area  production  yield  season_start_month  season_end_month  \
0  21400.0      2600.0  0.121                   6                10   
1   1400.0       500.0  0.357                   6                10   
2   1000.0       100.0  0.100                   6                10   
3   7300.0      9400.0  1.288                   6                10   
4   3700.0      7100.0  1.919                   6                10   

   seasonal_precip_total  seasonal_precip_avg  seas

## Verification Steps

Validate the three tables before saving.

In [17]:
print("VERIFICATION CHECKS")
print("=" * 80)

# 1. Check row counts
print("\n1. Row Counts:")
print(f"   dim_districts:         {len(dim_districts):>8,} rows (expected ~515)")
print(
    f"   fact_monthly_climate:  {len(fact_monthly_climate):>8,} rows (expected ~67,000)"
)
print(
    f"   fact_seasonal_crop:    {len(fact_seasonal_crop):>8,} rows (expected ~357,000)"
)

# 2. Check for duplicates
print("\n2. Duplicate Check:")
dim_dups = dim_districts.duplicated(["state", "district"]).sum()
monthly_dups = fact_monthly_climate.duplicated(
    ["state", "district", "year", "month"]
).sum()
seasonal_dups = fact_seasonal_crop.duplicated(
    ["state", "district", "crop_year", "crop_name", "season"]
).sum()

print(f"   dim_districts duplicates:        {dim_dups} (must be 0)")
print(f"   fact_monthly_climate duplicates: {monthly_dups} (must be 0)")
print(f"   fact_seasonal_crop duplicates:   {seasonal_dups} (must be 0)")

assert dim_dups == 0, "❌ Found duplicates in dim_districts!"
assert monthly_dups == 0, "❌ Found duplicates in fact_monthly_climate!"
assert seasonal_dups == 0, "❌ Found duplicates in fact_seasonal_crop!"
print("   ✓ No duplicates found")

# 3. Check coverage
print("\n3. Weather Data Coverage in Seasonal Crop:")
precip_coverage = fact_seasonal_crop["seasonal_precip_total"].notna().mean() * 100
ndvi_coverage = fact_seasonal_crop["seasonal_ndvi_mean"].notna().mean() * 100
print(f"   Precipitation coverage: {precip_coverage:>5.1f}%")
print(f"   NDVI coverage:          {ndvi_coverage:>5.1f}%")

# 4. Validate year-wrapping for Rabi
print("\n4. Rabi Year-Wrapping Validation:")
sample_rabi = fact_seasonal_crop[
    (fact_seasonal_crop["season"] == "Rabi")
    & (fact_seasonal_crop["crop_year"] == 2020)
    & (fact_seasonal_crop["seasonal_precip_total"].notna())
]

if len(sample_rabi) > 0:
    sample = sample_rabi.iloc[0]
    print(f"   Sample: {sample['state']}, {sample['district']}")
    print(f"   Rabi 2020 precipitation: {sample['seasonal_precip_total']:.1f} mm")
    print(f"   (Should be sum of Nov-Dec 2020 + Jan-Mar 2021)")
    print(f"   ✓ Year-wrapping appears correct")
else:
    print("   ⚠️  No Rabi 2020 records with precipitation found")

print("\n" + "=" * 80)
print("✓ All verification checks passed!")

VERIFICATION CHECKS

1. Row Counts:
   dim_districts:              515 rows (expected ~515)
   fact_monthly_climate:    55,587 rows (expected ~67,000)
   fact_seasonal_crop:     361,385 rows (expected ~357,000)

2. Duplicate Check:
   dim_districts duplicates:        0 (must be 0)
   fact_monthly_climate duplicates: 0 (must be 0)
   fact_seasonal_crop duplicates:   0 (must be 0)
   ✓ No duplicates found

3. Weather Data Coverage in Seasonal Crop:
   Precipitation coverage:  31.2%
   NDVI coverage:           33.7%

4. Rabi Year-Wrapping Validation:
   Sample: Andhra Pradesh, Anantapur
   Rabi 2020 precipitation: 63.1 mm
   (Should be sum of Nov-Dec 2020 + Jan-Mar 2021)
   ✓ Year-wrapping appears correct

✓ All verification checks passed!


## Save Three Tables

Save in both pickle (fast) and parquet (portable) formats.

In [ ]:
fact_seasonal_crop

In [20]:
fact_seasonal_crop.shape

(361385, 23)

In [21]:
print("Saving tables...\n")

output_path = pickle_folder_path

# Save as pickle
print("Saving pickle files...")
dim_districts.to_pickle(output_path / "dim_districts.pkl")
print(f"  ✓ dim_districts.pkl")

fact_monthly_climate.to_pickle(output_path / "fact_monthly_climate.pkl")
print(f"  ✓ fact_monthly_climate.pkl")

fact_seasonal_crop[:180000].to_pickle(output_path / "fact_seasonal_crop1.pkl")
print(f"  ✓ fact_seasonal_crop1.pkl")
fact_seasonal_crop[180000:].to_pickle(output_path / "fact_seasonal_crop2.pkl")
print(f"  ✓ fact_seasonal_crop2.pkl")

# Save as parquet
print("\nSaving parquet files...")
dim_districts.to_parquet(output_path / "dim_districts.parquet", index=False)
print(f"  ✓ dim_districts.parquet")

fact_monthly_climate.to_parquet(
    output_path / "fact_monthly_climate.parquet", index=False
)
print(f"  ✓ fact_monthly_climate.parquet")

fact_seasonal_crop.to_parquet(output_path / "fact_seasonal_crop.parquet", index=False)
print(f"  ✓ fact_seasonal_crop.parquet")

print("\n" + "=" * 80)
print("✓ All tables saved successfully!")
print(f"\nLocation: {output_path}")

Saving tables...

Saving pickle files...
  ✓ dim_districts.pkl
  ✓ fact_monthly_climate.pkl
  ✓ fact_seasonal_crop1.pkl
  ✓ fact_seasonal_crop2.pkl

Saving parquet files...
  ✓ dim_districts.parquet
  ✓ fact_monthly_climate.parquet
  ✓ fact_seasonal_crop.parquet

✓ All tables saved successfully!

Location: /Users/a0488240/Documents/Learning/MTech/project/dsba2026/data/processed


## Summary & Usage Examples

Three-table architecture created successfully!

In [22]:
print("SUMMARY")
print("=" * 80)
print("\nThree tables created:")
print(f"\n1. dim_districts ({len(dim_districts):,} rows)")
print("   Purpose: Geographic dimension with district centroids")
print("   Key: state, district")
print(f"   Columns: {list(dim_districts.columns)}")

print(f"\n2. fact_monthly_climate ({len(fact_monthly_climate):,} rows)")
print("   Purpose: Monthly weather + vegetation data")
print("   Key: state, district, year, month")
print(f"   Columns: {list(fact_monthly_climate.columns)}")

print(f"\n3. fact_seasonal_crop ({len(fact_seasonal_crop):,} rows)")
print("   Purpose: Seasonal crop yields + aggregated weather features")
print("   Key: state, district, crop_year, crop_name, season")
print(f"   Columns: {list(fact_seasonal_crop.columns)}")

print("\n" + "=" * 80)
print("\nUSAGE EXAMPLES:")
print("-" * 80)

print("\n# Use Case 1: ML Model Training")
print("df = pd.read_pickle('data/processed/fact_seasonal_crop1.pkl')")
print("df = pd.read_pickle('data/processed/fact_seasonal_crop2.pkl')")
print("features = ['seasonal_precip_total', 'seasonal_ndvi_mean', 'area']")
print("X = df[features]")
print("y = df['yield']")

print("\n# Use Case 2: Time-Series Analysis")
print("df = pd.read_pickle('data/processed/fact_monthly_climate.pkl')")
print("monthly_trends = df.groupby(['state', 'year', 'month']).agg({")
print("    'precipitation_mm': 'mean'")
print("}).reset_index()")

print("\n# Use Case 3: Add Coordinates to Any Analysis")
print("dim_districts = pd.read_pickle('data/processed/dim_districts.pkl')")
print(
    "df = df.merge(dim_districts[['state', 'district', 'centroid_lat', 'centroid_lon']],"
)
print("              on=['state', 'district'], how='left')")

print("\n" + "=" * 80)

SUMMARY

Three tables created:

1. dim_districts (515 rows)
   Purpose: Geographic dimension with district centroids
   Key: state, district
   Columns: ['district_id', 'state', 'district', 'centroid_lat', 'centroid_lon']

2. fact_monthly_climate (55,587 rows)
   Purpose: Monthly weather + vegetation data
   Key: state, district, year, month
   Columns: ['state', 'district', 'year', 'month', 'date', 'precipitation_mm', 'raw_ndvi_mean', 'crop_area_km2', 'valid_crop_area_km2', 'valid_crop_fraction', 'quality_status', 'data_source']

3. fact_seasonal_crop (361,385 rows)
   Purpose: Seasonal crop yields + aggregated weather features
   Key: state, district, crop_year, crop_name, season
   Columns: ['state', 'district', 'crop_year', 'crop_name', 'crop_type', 'season', 'area', 'production', 'yield', 'season_start_month', 'season_end_month', 'seasonal_precip_total', 'seasonal_precip_avg', 'seasonal_precip_max', 'rainy_days', 'seasonal_ndvi_mean', 'seasonal_ndvi_min', 'seasonal_ndvi_max', 'ear

## Next Steps

1. **Feature Engineering:** Use `fact_seasonal_crop` in `modeling/01_feature_engineering.ipynb`
2. **Time-Series Analysis:** Explore trends using `fact_monthly_climate`
3. **ML Modeling:** Train yield prediction models on `fact_seasonal_crop`
4. **Spatial Analysis:** Use `dim_districts` coordinates for mapping

See plan file for more details: `.claude/plans/given-the-four-dataframe-replicated-puppy.md`